# Can a model predict the transcriptome change of a *held-out* perturbation?

The seminar's final task (`04_`, following `02_condition_classification` and
`03_perturbation_clustering`): **predict the log2 fold-change transcriptome of knocking out a gene
that was never knocked out in the training data.**

- **50 modelling perturbations** + a **separate, disjoint 20-perturbation held-out test set**, both
  stratified across the Task 3 hierarchical clusters so each set spans the full range of effect types.
- **Control condition only** (as in Task 3): every knockout is measured against the same
  non-targeting `control` baseline, in the same experiment.
- Features describe the *target gene itself*, built from control cells only (never from its
  knockout), so they can be computed for a held-out gene.
- A **baseline** ("average knockout") plus **4 models**: {KNN, MLP} × {raw co-expression vector,
  + baseline scalars}.
- **Metric:** Pearson correlation of predicted vs observed log2FC. **Uncertainty:** a
  per-perturbation bootstrap confidence interval (resampling cells).

All computation is in standalone scripts under `perturbation_prediction/` (`data_prep.py`,
`train_baseline.py`, `train_knn.py`, `train_mlp.py`, `compare_models.py`); this notebook only reads
`perturbation_prediction/results/` back.

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import Image, display

R = "perturbation_prediction/results"
C = f"{R}/comparison"
pd.set_option("display.width", 140)

split = json.load(open(f"{R}/split_perturbations.json"))
targets = pd.read_csv(f"{R}/log2fc_targets.csv", index_col=0)
summary = pd.read_csv(f"{C}/summary_table.csv", index_col=0)

## 1. The data and the split

`data_prep.make_split`: start from the 239 perturbations that Task 3 gave a hierarchical-cluster
label, keep those with **≥50 Control cells** and a measured knocked-out gene (needed for a feature
vector), then sample **50 train + 20 test** stratified by cluster, preferring higher cell counts.

In [ ]:
tr = pd.Series([split["cluster_of"][g] for g in split["train"]]).value_counts().sort_index()
te = pd.Series([split["cluster_of"][g] for g in split["test"]]).value_counts().sort_index()
print("per Task-3 hierarchical cluster:  train", tr.to_dict(), "  test", te.to_dict())
print("cells/perturbation: median", int(np.median(list(split["n_cells"].values()))),
      " range", (min(split["n_cells"].values()), max(split["n_cells"].values())))
print()
print("TEST perturbations:", ", ".join(split["test"]))

## 2. The prediction target: pseudobulk log2 fold change

For perturbation $p$ and HVG $g$, over the Control-condition cells:

$$\text{log2FC}[p,g] = \log_2\!\frac{\bar{x}_{p,g} + \lambda}{\bar{x}_{\text{control},g} + \lambda}$$

where $\bar{x}$ is the mean library-normalised count (`expm1` of the stored log1p values) and
$\lambda = 0.3$ is a small prior count: without it, the many HVGs with very low control expression
give enormous noise-dominated ratios. Sanity check — a knockout should reduce its own transcript:

In [ ]:
ot = pd.Series({p: targets.loc[p, p] for p in split["train"] + split["test"] if p in targets.columns})
print(f"on-target log2FC over the {len(ot)} selected genes that are themselves HVGs:")
print(f"  median {ot.median():.2f} log2 units,  {(ot < 0).mean()*100:.0f}% negative")
print("  most reduced:", ", ".join(f"{g} ({v:.1f})" for g, v in ot.sort_values().head(5).items()))

## 3. Gene features — describing a gene you've never perturbed

Built by `data_prep.build_gene_features` from the **~15k non-targeting `control` cells only** (no
knockout cell enters feature construction, so a held-out gene is described exactly like a training
gene):

1. **Co-expression vector** (2000-d) — Pearson correlation of the target gene against each HVG across
   control cells. Genes co-regulated with gene *X* in normal cells are the ones expected to move when
   *X* is knocked out ("guilt by association").
2. **Baseline scalars** (23-d) — mean expression, detection fraction, dispersion, and the gene's
   correlation with each of the top-20 control-cell expression PCs.

`--features raw` uses (1) alone; `--features scalars` uses (1) ⊕ z-scored (2).

In [ ]:
d = np.load(f"{R}/gene_features.npz", allow_pickle=True)
ce = pd.DataFrame(d["coexp"], index=d["genes"], columns=d["hvg_names"])
ex = "IFNGR1" if "IFNGR1" in ce.index else ce.index[0]
print(f"{ce.shape[0]} genes with features   co-expression dim {ce.shape[1]}   scalars {list(d['scalar_names'][:3])}...")
print(f"\ntop co-expressed HVGs for {ex}:", ce.loc[ex].sort_values(ascending=False).head(8).round(2).to_dict())

## 4. Models

| name | features | algorithm |
|---|---|---|
| `baseline` | — | predict the **mean training log2FC** ("average knockout") — no features |
| `knn_raw` / `knn_scalars` | co-expr / +scalars | cosine k-NN over the 50 training perturbations, similarity-weighted mean of their observed log2FC; *k* by internal leave-one-out |
| `mlp_raw` / `mlp_scalars` | co-expr / +scalars | 5-net ensemble predicting a 20-d PCA latent of the log2FC, averaged |

Each model is fit on the 50 modelling perturbations and predicts the 20 held-out test perturbations.

## 5. Results — predicted vs observed log2FC

Two metrics per held-out perturbation: **Pearson r** (is the *pattern* right?) and **RMSE** (is the
*magnitude* right?). CIs on the mean are bootstrapped over the 20 perturbations.

In [ ]:
cols = ["features", "pearson_mean", "pearson_ci_lo", "pearson_ci_hi", "pearson_median",
        "rmse_mean", "rmse_ci_lo", "rmse_ci_hi", "rmse_median"]
print(summary[cols].round(3).to_string())
display(Image(filename=f"{C}/bar_pearson.png"))
display(Image(filename=f"{C}/bar_rmse.png"))

On **both** metrics every feature-based model ties the `baseline`: mean Pearson ≈ 0.77–0.78
(median ≈ 0.85; the mean is dragged down by a few barely-expressed non-coding targets like
`IDI2-AS1`, `LINC00518`, `CTPS1` at r ≈ 0.4), and mean RMSE ≈ 0.05 log2 units for all of them. The
95% CIs overlap the baseline's completely. Adding scalar features barely moves the needle; KNN and
MLP are indistinguishable from each other and from predicting the average knockout.

**Why the scores are decent yet the models don't beat the baseline:** the predicted-vs-observed
scatter shows they are carried by the response knockouts *share* — on-target knockdown plus strongly
co-regulated genes — which is the same for every perturbation and is exactly what the "average
knockout" predicts.

In [ ]:
best = json.load(open(f"{C}/summary.json"))["best_model"]
display(Image(filename=f"{C}/scatter_{best}.png"))

## 6. Uncertainty — per-perturbation bootstrap confidence intervals

For each held-out perturbation we resample its cells with replacement 1000×, recompute the observed
log2FC and both scores, and take the 2.5/97.5 percentiles (`evaluate_utils.bootstrap_ci`). The CI is
tight for well-powered perturbations (`HLA-A`, `RTP4`, `SLC22A18`: ~300 cells) and wide for the
low-count ones (`CTPS1`: 115 cells). For every perturbation, on both metrics, the baseline and best
model CIs overlap — the data does not support a claim that either is better.

In [ ]:
mt = pd.read_csv(f"{R}/{best}/metrics_test.csv", index_col=0)
mb = pd.read_csv(f"{R}/baseline/metrics_test.csv", index_col=0)
tab = pd.DataFrame({"n_cells": mt["n_cells"],
                    "baseline_r": mb["pearson"].round(2), f"{best}_r": mt["pearson"].round(2),
                    f"{best}_r_CI": [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(mt["pearson_ci_lo"], mt["pearson_ci_hi"])],
                    "baseline_rmse": mb["rmse"].round(3), f"{best}_rmse": mt["rmse"].round(3),
                    f"{best}_rmse_CI": [f"[{lo:.3f}, {hi:.3f}]" for lo, hi in zip(mt["rmse_ci_lo"], mt["rmse_ci_hi"])]})
print(tab.sort_values("n_cells", ascending=False).to_string())
display(Image(filename=f"{C}/perturbation_ci.png"))
display(Image(filename=f"{C}/perturbation_ci_rmse.png"))

## 7. Summary

1. **A model can predict a held-out perturbation's log2FC transcriptome fairly well** — mean Pearson
   ≈ 0.78, median ≈ 0.85, mean RMSE ≈ 0.05 log2 units over the 20 held-out perturbations.
2. **But it cannot beat the "average knockout" baseline** on either metric. Every feature-based model
   (KNN or MLP, raw co-expression or + scalars) scores within noise of predicting the mean training
   log2FC. The score comes from the response knockouts share (on-target knockdown + co-regulated
   genes), not from anything specific to the held-out gene.
3. **The simplest choices win / tie:** raw co-expression ≈ co-expression + scalars; KNN ≈ MLP;
   neither beats the baseline.
4. **The bootstrap CIs make the negative result rigorous:** per-perturbation CIs (Pearson *and* RMSE)
   are wide, especially for low-cell perturbations, and the baseline and best-model intervals overlap
   everywhere — there is no perturbation for which a feature-based model is reliably better.